# Interview Transcript Semantic Chunker

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

## What This Notebook Does

This notebook segments interview transcripts into semantically coherent chunks—units of text that hold together thematically rather than being split at arbitrary word counts. Using sentence-transformer embeddings, it measures topical similarity between consecutive sentences and places chunk boundaries where the topic shifts. The result is a set of meaningful segments ready for qualitative coding and thematic analysis. The notebook runs entirely locally: sentence embeddings are computed on the machine running the notebook, no API key is needed, and transcripts never leave the session.

Each chunk receives a coherence score (0–1) measuring how internally consistent it is, giving you immediate quality feedback without manual inspection.

## Key Features

- **Fully local processing** — no API key required; transcripts never leave your machine
- **Semantic chunking**: Embedding-based similarity detection finds natural topic boundaries, with a configurable similarity threshold and minimum/maximum sentences per chunk
- **Speaker-aware segmentation**: When enabled, chunks never mix speakers; very long single-speaker turns may still be divided by the maximum-sentences limit
- **Coherence scoring**: Each chunk is scored for internal semantic consistency, and low-coherence chunks are flagged for review
- **Similarity visualization**: Plot of sentence-to-sentence similarity scores with chunk boundaries marked
- **Multiple input formats**: PDF, DOCX, TXT transcript files
- **Export**: CSV and JSON with chunk ID, text, speaker, word count, coherence score, and research-context metadata for downstream integration with the Codebook Builder and Coding notebooks

## Workflow

1. Install dependencies and configure settings
2. Set research context (optional but recommended for downstream integration)
3. Upload transcript file(s)
4. Configure chunking parameters (threshold, sentence limits)
5. Run chunking and review the similarity visualization
6. Export chunked transcript for use in Coding and Thematic Analysis

## Applications

This notebook supports the preparation of interview corpora for qualitative coding, from dissertation fieldwork to applied research projects. Its outputs feed directly into the Coding and Thematic Analysis notebook and can also be imported into QDA software.

## Methodological Positioning

This notebook represents a computational anthropology approach in which computational segmentation supports rather than replaces interpretive judgment. The chunker prepares transcripts for analysis; it does not interpret them. Coherence scores flag chunks that may warrant researcher review, and every segmentation decision is documented in the export metadata, keeping the process transparent and reproducible.

## Target Audience

Designed for anthropologists and qualitative researchers working with interview transcripts; no programming experience is required.

## Technical Approach

The notebook encodes each sentence with a sentence-transformer model (all-MiniLM-L6-v2) and computes cosine similarity between consecutive sentences, placing chunk boundaries where similarity drops below a configurable threshold. Speaker-aware segmentation keeps each chunk within a single speaker's turn, and each chunk receives a coherence score computed from the mean pairwise similarity of its sentences.

## Contributing to AI Anthropology

This notebook is part of the AI Anthropology Toolkit, an effort to operationalize AI-assisted qualitative research. By developing and sharing computational approaches to traditional qualitative methods, the toolkit explores how machine learning can augment ethnographic practice while maintaining the interpretive depth that defines anthropological inquiry.

## AI Anthropology Toolkit

- **[Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Interview_Transcript_Semantic_Chunker.ipynb)**: Semantic segmentation of interview transcripts (this notebook)
- **[Qualitative Codebook Builder](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Qualitative_Codebook_Builder.ipynb)**: AI-assisted extraction of codes and themes from source documents
- **[Coding and Thematic Analysis](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Coding_and_Thematic_Analysis.ipynb)**: AI-assisted application of codebook frameworks to segmented transcripts

## License & Attribution

This work is licensed under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812


## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
# Install and import required packages

!pip install -q pypdf python-docx "nltk>=3.8,<4" pandas "sentence-transformers>=3,<6" scipy matplotlib

import os
import json
import re
from datetime import datetime
from typing import List, Tuple
from dataclasses import dataclass

import pandas as pd
import numpy as np
from pypdf import PdfReader
from docx import Document
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine as cosine_distance
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load sentence transformer model
print("Loading sentence transformer model...")
_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("All packages installed and imported successfully")

## Configuration

*Set up research context and chunking parameters.*

In [ ]:
# Configuration and data structures

@dataclass
class ChunkEntry:
    """A single chunk of transcript text."""
    chunk_id: int = 0
    text: str = ""
    speaker: str = ""
    start_sentence: int = 0
    end_sentence: int = 0
    word_count: int = 0
    coherence_score: float = 0.0  # mean intra-chunk sentence similarity (0-1)
    source_file: str = ""
    timestamp_start: str = ""
    timestamp_end: str = ""


class ChunkerConfig:
    """Configuration for the semantic chunker."""
    # Research Context
    PROJECT_NAME = ""
    RESEARCH_QUESTION = ""

    # Chunking Parameters
    SIMILARITY_THRESHOLD = 0.5  # Lower = more chunks (0.1-0.9)
    MAX_CHUNK_SENTENCES = 5
    MIN_CHUNK_SENTENCES = 1

    # Speaker Handling
    PRESERVE_SPEAKER_BOUNDARIES = True
    PRESERVE_TIMESTAMPS = True

    # Output
    OUTPUT_PATH = "/content/chunker_outputs/"


# Global state
uploaded_transcripts = {}

print("Configuration structures defined")

In [ ]:
# Interactive configuration interface

def create_chunker_config_interface():
    """Create configuration interface for the semantic chunker."""

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Configure Semantic Chunker</h3>
    <p>Set up your research context and chunking parameters. The chunker uses sentence embeddings to find
    natural topic boundaries in your transcripts.</p>
    <p style='color: #6096BA; font-size: 13px;'>The research question is optional but recommended for
    integration with the Codebook Builder and Coding notebooks.</p>
    </div>
    """

    style = {'description_width': '200px'}
    layout = widgets.Layout(width='500px')

    # Research Context
    research_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        Research Context
    </h3>
    """)

    project_name_widget = widgets.Text(
        value=ChunkerConfig.PROJECT_NAME,
        placeholder='e.g., "AI Governance Study"',
        description='Project Name:',
        style=style, layout=layout
    )

    research_question_widget = widgets.Textarea(
        value=ChunkerConfig.RESEARCH_QUESTION,
        placeholder='e.g., "How do healthcare practitioners negotiate AI-driven decision-making?"',
        description='Research Question:',
        style=style,
        layout=widgets.Layout(width='600px', height='70px')
    )

    # Chunking Parameters
    params_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        Chunking Parameters
    </h3>
    """)

    threshold_widget = widgets.FloatSlider(
        value=ChunkerConfig.SIMILARITY_THRESHOLD,
        min=0.1, max=0.9, step=0.05,
        description='Similarity Threshold:',
        style=style, layout=layout,
        readout_format='.2f'
    )
    threshold_help = widgets.HTML(
        "<span style='color: #6096BA; font-size: 12px; margin-left: 25px;'>Lower = more chunks (more topic splits). Higher = fewer, larger chunks.</span>"
    )

    max_sentences_widget = widgets.IntSlider(
        value=ChunkerConfig.MAX_CHUNK_SENTENCES,
        min=2, max=15, step=1,
        description='Max Sentences/Chunk:',
        style=style, layout=layout
    )

    min_sentences_widget = widgets.IntSlider(
        value=ChunkerConfig.MIN_CHUNK_SENTENCES,
        min=1, max=5, step=1,
        description='Min Sentences/Chunk:',
        style=style, layout=layout
    )

    preserve_speaker_widget = widgets.Checkbox(
        value=ChunkerConfig.PRESERVE_SPEAKER_BOUNDARIES,
        description='Preserve speaker boundaries (chunks never mix speakers)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='auto')
    )
    preserve_speaker_help = widgets.HTML(
        "<span style='color: #6096BA; font-size: 12px; margin-left: 25px;'>Very long single-speaker turns may still be divided by the max-sentences limit.</span>"
    )

    preserve_timestamps_widget = widgets.Checkbox(
        value=ChunkerConfig.PRESERVE_TIMESTAMPS,
        description='Preserve timestamps (if present in transcript)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='auto')
    )

    # Apply button
    apply_button = widgets.Button(
        description='Apply Configuration',
        icon='check',
        layout=widgets.Layout(width='200px', height='45px', margin='15px 0'),
        style={'button_color': '#6096BA', 'font_weight': 'bold'}
    )

    status_output = widgets.Output()

    def apply_config(b):
        with status_output:
            status_output.clear_output()

            ChunkerConfig.PROJECT_NAME = project_name_widget.value.strip()
            ChunkerConfig.RESEARCH_QUESTION = research_question_widget.value.strip()
            ChunkerConfig.SIMILARITY_THRESHOLD = threshold_widget.value

            min_sentences = min_sentences_widget.value
            max_sentences = max_sentences_widget.value
            if min_sentences > max_sentences:
                print(f"Note: minimum sentences per chunk ({min_sentences}) cannot exceed "
                      f"the maximum ({max_sentences}). Using {max_sentences} as the minimum.")
                min_sentences = max_sentences
            ChunkerConfig.MAX_CHUNK_SENTENCES = max_sentences
            ChunkerConfig.MIN_CHUNK_SENTENCES = min_sentences
            ChunkerConfig.PRESERVE_SPEAKER_BOUNDARIES = preserve_speaker_widget.value
            ChunkerConfig.PRESERVE_TIMESTAMPS = preserve_timestamps_widget.value

            os.makedirs(ChunkerConfig.OUTPUT_PATH, exist_ok=True)

            print("Configuration applied!")
            print(f"  Project: {ChunkerConfig.PROJECT_NAME or '(not set)'}")
            if ChunkerConfig.RESEARCH_QUESTION:
                rq = ChunkerConfig.RESEARCH_QUESTION
                rq_display = rq if len(rq) <= 100 else rq[:100] + "..."
                print(f"  Research Question: {rq_display}")
            print(f"  Similarity Threshold: {ChunkerConfig.SIMILARITY_THRESHOLD:.2f}")
            print(f"  Sentences per chunk: {ChunkerConfig.MIN_CHUNK_SENTENCES}-{ChunkerConfig.MAX_CHUNK_SENTENCES}")
            print(f"  Preserve speaker boundaries: {ChunkerConfig.PRESERVE_SPEAKER_BOUNDARIES}")

    apply_button.on_click(apply_config)

    # Single-column layout
    config_box = widgets.VBox([
        research_header,
        project_name_widget,
        research_question_widget,
        params_header,
        threshold_widget, threshold_help,
        max_sentences_widget, min_sentences_widget,
        preserve_speaker_widget, preserve_speaker_help, preserve_timestamps_widget,
    ])

    display(HTML(instructions_html))
    display(config_box)
    display(apply_button)
    display(status_output)

create_chunker_config_interface()

## Transcript Upload

*Upload interview transcript files. Supported formats: PDF, DOCX, TXT. The chunker will detect speaker labels (e.g., "Interviewer:", "P1:", "Dr. Smith:") and timestamps automatically.*

In [ ]:
# Transcript upload and text extraction with speaker/timestamp parsing

def extract_text_from_pdf(file_content: bytes) -> str:
    """Extract text from PDF file."""
    import io
    reader = PdfReader(io.BytesIO(file_content))
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text


def extract_text_from_docx(file_content: bytes) -> str:
    """Extract text from DOCX file, including table contents."""
    import io
    doc = Document(io.BytesIO(file_content))
    parts = [para.text for para in doc.paragraphs]
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    parts.append(cell.text)
    return "\n".join(parts)


# Common speaker label patterns. The delimiter is a colon only, and the
# capitalized catch-all is limited to one or two words (25 characters or
# fewer) immediately followed by ':' so that lines like "Note:" or
# "Background:" and mid-sentence colons do not create phantom speakers.
_SPEAKER_PATTERN = re.compile(
    r'^\s*'
    r'(?:'
    r'(?P<speaker>'
    r'(?:Interviewer|Interviewee|Respondent|Participant|Moderator|Speaker)'
    r'|(?:P|R|S|I|M)\d{0,3}'
    r'|(?:Dr|Mr|Mrs|Ms|Prof)\.?\s+[A-Z][a-z]+'
    r'|(?!(?i:note|notes|background|summary|question|answer|topic|date|time|location|source|transcript|warning|example)\b)'
    r'(?=[^:\n]{1,25}:)'
    r'(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?|[A-Z][A-Z\'\-]{1,24}(?:\s+[A-Z][A-Z\'\-]+)?)'
    r'(?=:)'
    r')'
    r')'
    r'\s*:\s*',
    re.MULTILINE
)

# Timestamp patterns: bracketed [00:12:34], parenthesized (12:34), or a bare
# HH:MM:SS at the start of a line. Bare MM:SS forms are not treated as
# timestamps, so a line beginning "3:30 in the afternoon" is left intact.
_TIMESTAMP_PATTERN = re.compile(
    r'(?:\[\s*(\d{1,2}:\d{2}(?::\d{2})?)\s*\]'
    r'|\(\s*(\d{1,2}:\d{2}(?::\d{2})?)\s*\)'
    r'|(\d{1,2}:\d{2}:\d{2})\b)'
)


@dataclass
class ParsedLine:
    """A single line from a transcript with extracted metadata."""
    text: str = ""
    speaker: str = ""
    timestamp: str = ""
    line_number: int = 0


def parse_transcript(raw_text: str) -> List[ParsedLine]:
    """
    Parse raw transcript text into structured lines with speaker labels and timestamps.
    Handles various transcript formats.
    """
    lines = raw_text.split('\n')
    parsed_lines = []
    current_speaker = ""

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        # Check for timestamp
        timestamp = ""
        ts_match = _TIMESTAMP_PATTERN.match(line)
        if ts_match:
            timestamp = next(g for g in ts_match.groups() if g)
            line = line[ts_match.end():].strip()

        # Check for speaker label
        speaker_match = _SPEAKER_PATTERN.match(line)
        if speaker_match:
            current_speaker = speaker_match.group('speaker').strip()
            line = line[speaker_match.end():].strip()

        if line:
            parsed_lines.append(ParsedLine(
                text=line,
                speaker=current_speaker,
                timestamp=timestamp,
                line_number=i + 1
            ))

    return parsed_lines


def create_upload_interface():
    """Create transcript upload interface."""
    global uploaded_transcripts

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Upload Interview Transcripts</h3>
    <p>Upload your transcript files. Supported formats: <strong>PDF, DOCX, TXT</strong></p>
    <p style='color: #6096BA; font-size: 13px;'>Speaker labels (e.g., \"Interviewer:\", \"P1:\") and timestamps will be detected automatically.</p>
    </div>
    """

    upload_button = widgets.Button(
        description='Upload Transcripts',
        icon='upload',
        layout=widgets.Layout(width='200px', height='50px'),
        style={'button_color': '#6096BA', 'font_weight': 'bold'}
    )

    clear_button = widgets.Button(
        description='Clear All',
        icon='trash',
        layout=widgets.Layout(width='150px', height='40px'),
        style={'button_color': '#8B8C89', 'font_weight': 'bold'}
    )

    status_output = widgets.Output()

    def handle_upload(b):
        with status_output:
            status_output.clear_output()
            print("Select files to upload...")

        try:
            uploaded = files.upload()
            with status_output:
                for filename, content in uploaded.items():
                    try:
                        if filename.lower().endswith('.pdf'):
                            text = extract_text_from_pdf(content)
                        elif filename.lower().endswith('.docx'):
                            text = extract_text_from_docx(content)
                        elif filename.lower().endswith('.txt'):
                            try:
                                text = content.decode('utf-8')
                            except UnicodeDecodeError:
                                text = content.decode('latin-1')
                                print(f"  Note: {filename} is not UTF-8; decoded as Latin-1 instead.")
                        else:
                            print(f"Unsupported format: {filename}")
                            continue

                        parsed = parse_transcript(text)
                        uploaded_transcripts[filename] = {
                            'raw_text': text,
                            'parsed_lines': parsed
                        }

                        # Summary
                        speakers = set(l.speaker for l in parsed if l.speaker)
                        has_timestamps = any(l.timestamp for l in parsed)
                        word_count = sum(len(l.text.split()) for l in parsed)

                        print(f"{filename}:")
                        print(f"  Words: {word_count:,} | Lines: {len(parsed)}")
                        if speakers:
                            print(f"  Speakers detected: {', '.join(sorted(speakers))}")
                        else:
                            print(f"  No speaker labels detected")
                        if has_timestamps:
                            print(f"  Timestamps: detected")

                    except Exception as e:
                        print(f"Error processing {filename}: {e}")

                print(f"\nTotal transcripts loaded: {len(uploaded_transcripts)}")

        except Exception as e:
            with status_output:
                print(f"Upload cancelled or failed: {e}")

    def handle_clear(b):
        global uploaded_transcripts
        uploaded_transcripts = {}
        with status_output:
            status_output.clear_output()
            print("All transcripts cleared")

    upload_button.on_click(handle_upload)
    clear_button.on_click(handle_clear)

    display(HTML(instructions_html))
    display(widgets.HBox([upload_button, clear_button]))
    display(status_output)

create_upload_interface()

## Semantic Chunking Engine

*Embedding-based chunking using sentence-transformer similarity scores. Chunk boundaries are placed where topic similarity drops below the configured threshold, with speaker boundary preservation and coherence scoring.*

In [ ]:
# Semantic chunking engine

def compute_sentence_similarities(sentences: List[str]) -> Tuple[List[float], np.ndarray]:
    """
    Encode all sentences in a single pass and compute cosine similarity
    between each consecutive pair. Returns (similarities, embeddings):
    N-1 similarity scores plus the N sentence embeddings, which are
    reused for coherence scoring.
    """
    if not sentences:
        return [], np.empty((0, 0))

    embeddings = _embedding_model.encode(sentences, show_progress_bar=True)
    similarities = []

    for i in range(len(embeddings) - 1):
        sim = 1.0 - cosine_distance(embeddings[i], embeddings[i + 1])
        similarities.append(sim)

    return similarities, embeddings


def compute_chunk_coherence(chunks: List[ChunkEntry], embeddings: np.ndarray) -> None:
    """
    Compute coherence score for each chunk by indexing into the sentence
    embeddings already computed for similarity scoring (no re-encoding).
    Coherence = mean pairwise cosine similarity of sentences within the chunk.
    Higher = more internally consistent. Modifies chunks in place.
    """
    for chunk in chunks:
        start = chunk.start_sentence
        end = chunk.end_sentence + 1
        chunk_embeddings = embeddings[start:end]

        if len(chunk_embeddings) < 2:
            chunk.coherence_score = 1.0  # single sentence is perfectly coherent
            continue

        sims = []
        for i in range(len(chunk_embeddings)):
            for j in range(i + 1, len(chunk_embeddings)):
                sim = 1.0 - cosine_distance(chunk_embeddings[i], chunk_embeddings[j])
                sims.append(sim)

        chunk.coherence_score = round(float(np.mean(sims)), 3) if sims else 1.0


def chunk_by_embedding(parsed_lines: List[ParsedLine],
                       threshold: float,
                       max_sentences: int,
                       min_sentences: int,
                       preserve_speakers: bool) -> Tuple[List[ChunkEntry], List[float]]:
    """
    Embedding-based semantic chunking.

    Computes sentence-to-sentence similarity. Places chunk boundaries where
    similarity drops below threshold, respecting min/max sentence constraints
    and optionally preserving speaker boundaries.

    Returns: (chunks, similarity_scores)
    """
    if not parsed_lines:
        return [], []

    # Build sentences from parsed lines (each line may have multiple sentences)
    sentences = []
    sentence_metadata = []  # Track speaker/timestamp per sentence

    for pl in parsed_lines:
        line_sents = sent_tokenize(pl.text)
        for s in line_sents:
            sentences.append(s)
            sentence_metadata.append({
                'speaker': pl.speaker,
                'timestamp': pl.timestamp,
                'line_number': pl.line_number
            })

    if not sentences:
        return [], []

    # Compute similarities (single embedding pass, reused for coherence)
    similarities, embeddings = compute_sentence_similarities(sentences)

    # Find chunk boundaries. The size counter is incremented at the top of
    # each iteration, so it always equals the number of sentences currently
    # in the chunk when the split decision is made.
    boundaries = []
    current_chunk_start = 0
    current_chunk_size = 0

    for i, sim in enumerate(similarities):
        next_idx = i + 1
        current_chunk_size += 1

        should_split = False

        # Speaker boundary split: chunks never mix speakers, regardless of
        # the minimum-sentences setting
        if (preserve_speakers and
                next_idx < len(sentence_metadata) and
                sentence_metadata[next_idx]['speaker'] and
                sentence_metadata[next_idx]['speaker'] != sentence_metadata[i]['speaker']):
            should_split = True

        # Forced split: max sentences reached
        elif current_chunk_size >= max_sentences:
            should_split = True

        # Similarity-based split: topic shift detected
        elif sim < threshold and current_chunk_size >= min_sentences:
            should_split = True

        if should_split:
            boundaries.append(next_idx)
            current_chunk_start = next_idx
            current_chunk_size = 0

    # Build chunks from boundaries
    chunk_starts = [0] + boundaries
    chunk_ends = boundaries + [len(sentences)]

    chunks = []
    for chunk_id, (start, end) in enumerate(zip(chunk_starts, chunk_ends)):
        chunk_sentences = sentences[start:end]
        chunk_text = ' '.join(chunk_sentences)

        # Determine speaker for this chunk (most common speaker)
        chunk_speakers = [sentence_metadata[i]['speaker'] for i in range(start, end) if sentence_metadata[i]['speaker']]
        primary_speaker = max(set(chunk_speakers), key=chunk_speakers.count) if chunk_speakers else ''

        # Timestamps
        ts_start = sentence_metadata[start]['timestamp'] if sentence_metadata[start]['timestamp'] else ''
        ts_end = sentence_metadata[end - 1]['timestamp'] if sentence_metadata[end - 1]['timestamp'] else ''

        chunks.append(ChunkEntry(
            chunk_id=chunk_id + 1,
            text=chunk_text,
            speaker=primary_speaker,
            start_sentence=start,
            end_sentence=end - 1,
            word_count=len(chunk_text.split()),
            timestamp_start=ts_start,
            timestamp_end=ts_end
        ))

    # Compute coherence scores from the shared embeddings
    compute_chunk_coherence(chunks, embeddings)

    return chunks, similarities


def run_chunking(filename: str) -> Tuple[List[ChunkEntry], List[float]]:
    """
    Run semantic chunking on a transcript.
    Returns: (chunks, similarity_scores)
    """
    if filename not in uploaded_transcripts:
        print(f"Transcript not found: {filename}")
        return [], []

    parsed_lines = uploaded_transcripts[filename]['parsed_lines']

    print(f"Chunking '{filename}'...")
    print(f"  Lines: {len(parsed_lines)} | Threshold: {ChunkerConfig.SIMILARITY_THRESHOLD:.2f}")

    chunks, similarities = chunk_by_embedding(
        parsed_lines,
        ChunkerConfig.SIMILARITY_THRESHOLD,
        ChunkerConfig.MAX_CHUNK_SENTENCES,
        ChunkerConfig.MIN_CHUNK_SENTENCES,
        ChunkerConfig.PRESERVE_SPEAKER_BOUNDARIES
    )

    # Attach source filename
    for chunk in chunks:
        chunk.source_file = filename

    # Summary
    if chunks:
        word_counts = [c.word_count for c in chunks]
        coherence_scores = [c.coherence_score for c in chunks]
        low_coherence = [c for c in chunks if c.coherence_score < 0.5]
        speakers = set(c.speaker for c in chunks if c.speaker)

        print(f"\nChunking complete:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Words per chunk: min={min(word_counts)}, max={max(word_counts)}, avg={np.mean(word_counts):.0f}")
        print(f"  Mean coherence: {np.mean(coherence_scores):.2f}")
        if low_coherence:
            print(f"  Low coherence chunks (<0.5): {len(low_coherence)} — may warrant manual review")
        if speakers:
            print(f"  Speakers: {', '.join(sorted(speakers))}")
    else:
        print("No chunks produced.")

    return chunks, similarities


print("Semantic chunking engine loaded")

## Chunking Preview and Visualization

*Preview the chunking results with a similarity score plot showing where chunk boundaries were placed. Review individual chunks with their speaker labels and word counts.*

In [ ]:
# Chunking preview and similarity visualization

def plot_similarity_scores(similarities: List[float], chunks: List[ChunkEntry],
                           threshold: float, filename: str = ""):
    """
    Plot sentence-to-sentence similarity scores with chunk boundaries marked.
    """
    if not similarities:
        print("No similarity scores to plot.")
        return

    fig, ax = plt.subplots(figsize=(14, 5))

    x = range(len(similarities))
    ax.plot(x, similarities, color='#6096BA', linewidth=1.5, alpha=0.8, label='Similarity')
    ax.fill_between(x, similarities, alpha=0.15, color='#6096BA')

    # Threshold line
    ax.axhline(y=threshold, color='#274C77', linestyle='--', alpha=0.6,
               label=f'Threshold ({threshold:.2f})')

    # Mark chunk boundaries
    boundary_positions = [c.start_sentence for c in chunks if c.start_sentence > 0]
    for bp in boundary_positions:
        if bp - 1 < len(similarities):
            ax.axvline(x=bp - 1, color='#274C77', linestyle='--', alpha=0.4, linewidth=1.5)

    ax.set_xlabel('Sentence Index', fontsize=11)
    ax.set_ylabel('Cosine Similarity', fontsize=11)
    title = 'Sentence-to-Sentence Similarity with Chunk Boundaries'
    if filename:
        title += f'\n{filename}'
    ax.set_title(title, fontsize=13, color='#274C77')
    ax.set_ylim(-0.1, 1.1)
    ax.legend(loc='upper right')

    # Add boundary count annotation
    boundary_patch = mpatches.Patch(color='#274C77', alpha=0.4,
                                    label=f'Chunk boundaries ({len(boundary_positions)})')
    ax.legend(handles=[ax.get_legend().legend_handles[0],
                       ax.get_legend().legend_handles[1],
                       boundary_patch], loc='upper right')

    plt.tight_layout()
    plt.show()


def preview_chunks(chunks: List[ChunkEntry], max_display: int = 10):
    """
    Display a preview of the chunking results with coherence scores.
    """
    if not chunks:
        print("No chunks to preview.")
        return

    mean_coherence = np.mean([c.coherence_score for c in chunks])
    preview_html = f"""
    <div style='background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Chunk Preview ({len(chunks)} total chunks | mean coherence: {mean_coherence:.2f})</h3>
    <p style='color: #6096BA;'>Showing first {min(max_display, len(chunks))} chunks</p>
    </div>
    """
    display(HTML(preview_html))

    for chunk in chunks[:max_display]:
        speaker_tag = f" [{chunk.speaker}]" if chunk.speaker else ""
        ts_tag = ""
        if chunk.timestamp_start:
            ts_tag = f" ({chunk.timestamp_start}"
            if chunk.timestamp_end and chunk.timestamp_end != chunk.timestamp_start:
                ts_tag += f" - {chunk.timestamp_end}"
            ts_tag += ")"

        coherence_indicator = ""
        if chunk.coherence_score < 0.5:
            coherence_indicator = " ⚠"

        print(f"--- Chunk {chunk.chunk_id}{speaker_tag}{ts_tag} | {chunk.word_count} words | coherence: {chunk.coherence_score:.2f}{coherence_indicator} ---")
        preview_text = chunk.text[:200]
        if len(chunk.text) > 200:
            preview_text += "..."
        print(preview_text)
        print()


print("Visualization functions loaded")

## Run Chunking

*Execute the chunking process on your uploaded transcripts. Results are displayed with a similarity visualization and chunk preview.*

In [ ]:
# Run chunking on all uploaded transcripts

all_chunks = {}  # filename -> list of ChunkEntry
all_similarities = {}  # filename -> list of similarity scores

if not uploaded_transcripts:
    print("No transcripts uploaded. Please upload transcript files first.")
else:
    for filename in uploaded_transcripts:
        print("=" * 60)
        chunks, similarities = run_chunking(filename)
        all_chunks[filename] = chunks
        all_similarities[filename] = similarities

        # Visualize
        plot_similarity_scores(
            similarities, chunks,
            ChunkerConfig.SIMILARITY_THRESHOLD,
            filename
        )

        # Preview
        preview_chunks(chunks, max_display=5)
        print()

## Export Chunked Transcripts

*Export the chunked transcripts as CSV files ready for use in the Coding and Thematic Analysis notebook. Each chunk includes its ID, text, speaker, word count, and source file metadata.*

In [ ]:
# Export chunked transcripts

def export_chunks_csv(chunks: List[ChunkEntry], filename: str):
    """Export chunks as CSV with coherence scores."""
    rows = []
    for chunk in chunks:
        rows.append({
            'chunk_id': chunk.chunk_id,
            'text': chunk.text,
            'speaker': chunk.speaker,
            'word_count': chunk.word_count,
            'coherence_score': chunk.coherence_score,
            'start_sentence': chunk.start_sentence,
            'end_sentence': chunk.end_sentence,
            'timestamp_start': chunk.timestamp_start,
            'timestamp_end': chunk.timestamp_end,
            'source_file': chunk.source_file
        })

    df = pd.DataFrame(rows)
    base_name = os.path.splitext(filename)[0]
    output_path = f"{ChunkerConfig.OUTPUT_PATH}{base_name}_chunked.csv"
    df.to_csv(output_path, index=False)
    print(f"  {base_name}_chunked.csv ({len(chunks)} chunks)")
    return output_path


def export_chunks_json(chunks: List[ChunkEntry], filename: str):
    """Export chunks as JSON with metadata and coherence scores."""
    export_data = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'source_file': filename,
            'project_name': ChunkerConfig.PROJECT_NAME,
            'research_question': ChunkerConfig.RESEARCH_QUESTION,
            'similarity_threshold': ChunkerConfig.SIMILARITY_THRESHOLD,
            'max_chunk_sentences': ChunkerConfig.MAX_CHUNK_SENTENCES,
            'min_chunk_sentences': ChunkerConfig.MIN_CHUNK_SENTENCES,
            'preserve_speaker_boundaries': ChunkerConfig.PRESERVE_SPEAKER_BOUNDARIES,
            'total_chunks': len(chunks),
            'mean_coherence': round(float(np.mean([c.coherence_score for c in chunks])), 3) if chunks else 0.0
        },
        'chunks': [
            {
                'chunk_id': c.chunk_id,
                'text': c.text,
                'speaker': c.speaker,
                'word_count': c.word_count,
                'coherence_score': c.coherence_score,
                'start_sentence': c.start_sentence,
                'end_sentence': c.end_sentence,
                'timestamp_start': c.timestamp_start,
                'timestamp_end': c.timestamp_end,
                'source_file': c.source_file
            }
            for c in chunks
        ]
    }

    base_name = os.path.splitext(filename)[0]
    output_path = f"{ChunkerConfig.OUTPUT_PATH}{base_name}_chunked.json"
    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    print(f"  {base_name}_chunked.json")
    return output_path


def export_all_chunks():
    """Export all chunked transcripts."""
    if not all_chunks:
        print("No chunks to export. Run chunking first.")
        return

    os.makedirs(ChunkerConfig.OUTPUT_PATH, exist_ok=True)

    print("Exporting chunked transcripts...")
    for filename, chunks in all_chunks.items():
        if chunks:
            export_chunks_csv(chunks, filename)
            export_chunks_json(chunks, filename)
        else:
            print(f"  {filename}: no chunks to export")

    print(f"\nAll exports saved to: {ChunkerConfig.OUTPUT_PATH}")
    print("\nThese files can be loaded directly into the Coding and Thematic Analysis notebook.")


export_all_chunks()

## Output Files

*View the generated chunked transcript files. Access them from the Colab file browser on the left side, then right-click to download.*

In [ ]:
# Output file listing

def show_output_files():
    """Display list of generated output files."""
    output_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Output Files</h3>
    <p>Your chunked transcripts are ready for use in the Coding and Thematic Analysis notebook.</p>
    <ul style='color: #274C77;'>
        <li><strong>CSV</strong>: [filename]_chunked.csv \u2014 For spreadsheet viewing and Coding notebook import</li>
        <li><strong>JSON</strong>: [filename]_chunked.json \u2014 With full metadata for programmatic use</li>
    </ul>
    </div>
    """
    display(HTML(output_html))

    print("Generated files:")
    if os.path.exists(ChunkerConfig.OUTPUT_PATH):
        for f in sorted(os.listdir(ChunkerConfig.OUTPUT_PATH)):
            filepath = os.path.join(ChunkerConfig.OUTPUT_PATH, f)
            size = os.path.getsize(filepath)
            if size > 1024:
                size_str = f"{size/1024:.1f} KB"
            else:
                size_str = f"{size} bytes"
            print(f"   {f} ({size_str})")
    else:
        print("   No output directory found. Run chunking and export first.")

show_output_files()

## Download Your Results

*Download the exported CSV and JSON files directly to your computer. If a download does not start, the files remain available in the Colab file browser at the paths printed below.*


In [ ]:
# Download exported files

def download_output_files():
    """Offer one-click download of the exported CSV/JSON files."""
    if not os.path.exists(ChunkerConfig.OUTPUT_PATH):
        print("No output directory found. Run chunking and export first.")
        return

    output_files = sorted(
        f for f in os.listdir(ChunkerConfig.OUTPUT_PATH)
        if f.endswith(('.csv', '.json'))
    )
    if not output_files:
        print("No exported files found. Run chunking and export first.")
        return

    print(f"Downloading {len(output_files)} file(s)...")
    for f in output_files:
        filepath = os.path.join(ChunkerConfig.OUTPUT_PATH, f)
        try:
            files.download(filepath)
            print(f"  {f}")
        except Exception as e:
            print(f"  Could not start download for {f}: {e}")
            print(f"  File available at: {filepath}")

download_output_files()
